In [1]:
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt

    
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import os

#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std,
                                          save_calibration_data, get_footprints_from_suite2p)

#
from utils.utils import smooth_ca_time_series, compute_dff0
from utils.calcium import calcium


Operating system:  Linux


Autosaving every 180 seconds


2023-07-25 15:51:43.971660: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-25 15:51:44.071533: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-07-25 15:51:44.074363: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/cat/.conda/envs/bmi/lib/python3.8/site-packages/cv2/../../lib64:
2023-07-25 1

In [2]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################

#
fname = '/media/cat/4TB/donato/andres/DON-012503/day0/calibration/Image_001_001.raw'

#
#fname = r'F:\bmi\cohort11\DON-015962\day0\calibration\Image_001_001.raw'

# 
bmi_c = CalibrationTools(fname)
bmi_c.calcium = calcium
#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is fast in other implemenations

#
bmi_c.std_map = bmi_c.data[:1000].mean(0)

# load suite2p footprints
bmi_c = get_footprints_from_suite2p(bmi_c)

# 
print ("DONE...")

memmap :  (90000, 512, 512)
  Fluorescence data loading information
         sample rate:  30 hz
         self.F (fluorescence):  (332, 90000)
         self.Fneu (neuropile):  (332, 90000)
         self.iscell (Suite2p cell classifier output):  (382, 2)
              of which number of good cells:  (332,)
         self.spks (deconnvoved spikes):  (332, 90000)
         self.stat (footprints structure):  (332,)
         mean std over all cells :  32.0
# of footprints;  332
DONE...


In [8]:
##########################################################
########### SELECT ENSEMBEL CELLS MANUALLY ###############
##########################################################
#
bmi_c.vmin = 300
bmi_c.vmax = 1000
bmi_c.select_cells()        
        

100%|██████████| 332/332 [00:05<00:00, 66.27it/s] 


previous selected cell for roi:  1  is:  nan
previous selected cell for roi:  0  is:  0.0
previous selected cell for roi:  1  is:  52.0


In [14]:
###############################################################
########### SHOW ENSEMBEL CELLS FROM SCRACTH AGAIN ############
###############################################################

#
print ("...THIS CELL MIGHT NOTE BE NECESSARY ...")

# convert to int
bmi_c.ensemble_ids = bmi_c.ensemble_ids.astype('int32')


# save ensemble rois
bmi_c.ensemble1 = [bmi_c.ensemble_ids[0],
                   bmi_c.ensemble_ids[1],]
bmi_c.ensemble2 = [bmi_c.ensemble_ids[2],
                    bmi_c.ensemble_ids[3]]

#
bmi_c.both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", bmi_c.both)

#
bmi_c.trace_subsample = 1        # Subsample the time series to go faster; PLEASE KEEP AT 1

# visualize traces
bmi_c.compute_traces_ensembles_Day0(bmi_c.std_map)

print ("DONE...")


.
all cells: [ 2 52 94 77]


100%|██████████| 90000/90000 [00:06<00:00, 13475.91it/s]


cell ids:  [ 2 52 94 77]
DONE...


In [13]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# if using binning
bmi_c.binning_flag = True
bmi_c.binning_time = 0.200
bmi_c.smoothing_n_bins = 3

# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 3   # reward lockout in seconds
bmi_c.post_missed_reward_lockout = 3
bmi_c.post_reward_lockout_baseline_min = .5 # want to wait until we drop to 30% of threshold <-------------------------------
bmi_c.trial_time = 30
                                 # TODO: in future load/save this to disk so that BMI 
                                 #   - doesn't use differetn lockout than calibration step
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
# find 30% reward threshold
bmi_c.reward_rate = 0.25#*0.85

#gr.find_reward_thresholds_low_and_high()
#bmi_c.high = 2
stepper = 0.85
bmi_c.find_reward_thresholds_high_realtime(stepper)  # this only rewards when sound passes specific level

#
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()


####################################
# do not change this
bmi_c.reward_rate_scaling_factor = 1.0

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)
print ("...DONE...")

nsec recording:  3000 max # of random rewards (i.e. every 30sec)  100
 @0.25% reward rate:  25
 high guess:  3.267297646399873
updated rewards #:  1  for threshold:  3.267297646399873
updated rewards #:  1  for threshold:  2.7772029994398917
updated rewards #:  2  for threshold:  2.360622549523908
updated rewards #:  3  for threshold:  2.0065291670953216
updated rewards #:  8  for threshold:  1.7055497920310232
updated rewards #:  13  for threshold:  1.4497173232263696
updated rewards #:  17  for threshold:  1.232259724742414
updated rewards #:  18  for threshold:  1.047420766031052
updated rewards #:  21  for threshold:  0.8903076511263941
updated rewards #:  22  for threshold:  0.756761503457435
updated rewards #:  24  for threshold:  0.6432472779388198
updated rewards #:  25  for threshold:  0.5467601862479968
updated rewards #:  28  for threshold:  0.4647461583107973
updated rewards #:  25  for threshold:  0.5467601862479968
FINAL # of rewards #:  25 , set threshold:  0.54676018624

In [15]:
#############################################
########### SAVE DATA #######################
#############################################

#    
text = "day0"
bmi_c = save_calibration_data(bmi_c,text)  
print ("...DONE...")

contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 


npyio.py (716): Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.


 couldn't save bmi_c.object .... TO FIX!
Done...
...DONE...
